# 🪐 GeoTwin-X // Cloud Host on Google Colab
### **Run Full Aerospace Backend & 3D Frontend on Cloud Hardware**

This notebook runs the **FastAPI server** (`backend/app.py`), binds it directly to the real **Jezero Crater CTX DEM** (`jezero_ready_20m.tif`) and the **Earth Global Grid Index** (`earth_global_grid_index.json`), and serves it over a **temporary preview URL** using either:
1. **Google Colab's Native Kernel Port Proxy** (instant, built-in to Colab)
2. **Cloudflare Quick Tunnel** (temporary `https://*.trycloudflare.com` URL with zero sign-up required)

--- 
## 📦 Step 1: Install Dependencies
Install FastAPI, Uvicorn, Rasterio, Scikit-Learn, and scientific libraries on the Colab runtime.

In [ ]:
!pip install -q fastapi uvicorn rasterio scipy scikit-learn

--- 
## 📂 Step 2: Unpack or Initialize Project Codebase
If you uploaded `geotwin-x.zip` to the Colab files panel on the left, this cell will unzip it automatically. If not, it will build the directory and compile the datasets directly!

In [ ]:
import os
import zipfile

if os.path.exists("geotwin-x.zip"):
    print("Unzipping geotwin-x.zip...")
    with zipfile.ZipFile("geotwin-x.zip", 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Unzipped successfully!")
elif os.path.exists("geotwin-x"):
    print("geotwin-x directory already present.")
else:
    print("Tip: Upload geotwin-x.zip to Colab's Files panel (folder icon on the left)!")

# Ensure datasets exist; if not, compile them with build_datasets.py
db_file = "geotwin-x/data/earth_analogs/earth_global_grid_index.json"
dem_file = "geotwin-x/data/mars_targets/jezero_ready_20m.tif"

if not (os.path.exists(db_file) and os.path.exists(dem_file)):
    print("Compiling real Mars DEM and Earth vector index...")
    !python geotwin-x/backend/build_datasets.py

print("Project assets ready:")
print(" - Earth DB:", os.path.exists(db_file))
print(" - Mars DEM:", os.path.exists(dem_file))

--- 
## 🚀 Step 3: Launch FastAPI Backend in Background
Start the Uvicorn ASGI server on `http://127.0.0.1:8000` as a background process.

In [ ]:
import subprocess
import time
import sys
import os

# Terminate any existing server process on port 8000
!fuser -k 8000/tcp || true

# Start FastAPI app using uvicorn in the background
server_process = subprocess.Popen(
    [
        sys.executable, "-m", "uvicorn", "backend.app:app",
        "--host", "127.0.0.1", "--port", "8000", "--app-dir", "geotwin-x"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(3)
print("FastAPI backend status: RUNNING (PID:", server_process.pid, ")")

--- 
## 🌐 Step 4: Expose Temporary Preview URL
You can view the dashboard using **Option A** (Google Colab's native proxy) or **Option B** (Cloudflare temporary public link).

In [ ]:
# ==========================================================================
# OPTION A: Google Colab Native Port Proxy (Click link below!)
# ==========================================================================
from google.colab.output import serve_kernel_port_as_window, serve_kernel_port_as_iframe

print("\n--- OPTION A: GOOGLE COLAB NATIVE PROXY ---")
serve_kernel_port_as_window(8000, anchor_text="👉 Click here to open GeoTwin-X Dashboard (Native Colab Window)")

In [ ]:
# ==========================================================================
# OPTION B: Cloudflare Quick Tunnel (Free Temporary URL, No Sign-up Needed)
# ==========================================================================
!fuser -k 8000/tcp || true # ensure running
!wget -q -c -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

import subprocess, time, re

# Ensure uvicorn is running
if server_process.poll() is not None:
    server_process = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "backend.app:app", "--host", "127.0.0.1", "--port", "8000", "--app-dir", "geotwin-x"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    time.sleep(2)

# Launch cloudflared tunnel in background and capture the URL
tunnel_proc = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

public_url = None
start_time = time.time()
while time.time() - start_time < 15:
    line = tunnel_proc.stderr.readline()
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    print("\n" + "="*70)
    print("  🪐 GeoTwin-X Temporary Cloudflare URL:")
    print("  👉 ", public_url)
    print("="*70 + "\n")
else:
    print("Could not extract trycloudflare URL. Please use Option A (Colab Native Proxy) above.")

--- 
## 🔗 Option C: Fixed Subdomain URL (Localtunnel)
Keeps the **same custom public link** every time you run this notebook. Perfect for live demos and team presentations.

In [ ]:
!npm install -g -q localtunnel

import urllib.request, subprocess, time

public_ip = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode("utf8").strip()

MY_SUBDOMAIN = "geotwin-x-nasa"

lt_proc = subprocess.Popen(["lt", "--port", "8000", "--subdomain", MY_SUBDOMAIN], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(3)

print("=" * 70)
print("🌐 FIXED PUBLIC URL: https://" + MY_SUBDOMAIN + ".loca.lt")
print("🔑 TUNNEL PASSWORD / IP (Enter on first page load):", public_ip)
print("=" * 70)

--- 
## 🛑 Step 5: Stop the Server When Finished
Run this cell when you want to stop the background FastAPI server.

In [ ]:
!fuser -k 8000/tcp
print("Server stopped successfully.")